# CellWarden quick start

[CellWarden](https://github.com/cellwarden/cellwarden) runs a full Seurat-based single-cell
RNA-seq analysis, from QC to annotation, differential expression and Open Targets evidence. LLM
agents propose decisions at four review gates, a reviewer agent checks them, and you approve.

This notebook has two parts:

1. **Look and feel (about 2 minutes):** the report CellWarden made for the 10x PBMC 3k dataset,
   and a speed demo of its Rust engine, **scfast**.
2. **Run it yourself (optional, about 25–35 minutes):** installs R / Seurat and CellWarden on
   this Colab machine and runs the whole pipeline on PBMC 3k.

Run the cells in order (Shift + Enter). No GPU is needed.

## Part 1a: the PBMC 3k report

This is the self-contained report from a CellWarden run on PBMC 3k (2,700 cells), with every
review gate on `auto`. Scroll through QC, clusters, markers, cell types, differential
expression, Open Targets evidence and the gate decisions.

In [ ]:
from IPython.display import HTML, IFrame, display

EXAMPLE = "https://cellwarden.github.io/cellwarden/examples/pbmc3k/"
display(HTML(
    f'<p>Open in a new tab: <a href="{EXAMPLE}report.html" target="_blank">report</a> · '
    f'<a href="{EXAMPLE}explorer.html" target="_blank">interactive UMAP</a> · '
    f'<a href="{EXAMPLE}dashboard.html" target="_blank">run dashboard</a></p>'))
IFrame(EXAMPLE + "report.html", width="100%", height=650)

## Part 1b: scfast speed demo

scfast is CellWarden's Rust engine. It does what Seurat's `NormalizeData`,
`FindVariableFeatures`, `ScaleData` + `RunPCA`, `FindNeighbors`, `FindClusters` and `RunUMAP`
do, giving the same principal components and neighbours.

In [ ]:
CW_VERSION = "0.3.0"
RELEASE = f"https://github.com/cellwarden/cellwarden/releases/download/v{CW_VERSION}"
!pip -q install "{RELEASE}/scfast-{CW_VERSION}-cp310-abi3-manylinux_2_28_x86_64.whl"
!test -d filtered_gene_bc_matrices/hg19 || curl -fsSL https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz | tar -xz

In [ ]:
import time
import numpy as np, scipy.io, scipy.sparse as sp
import matplotlib.pyplot as plt
import scfast

counts = sp.csr_matrix(scipy.io.mmread("filtered_gene_bc_matrices/hg19/matrix.mtx").T.astype(np.float32))  # cells x genes
counts.sum_duplicates()
times = {}

def step(name, fn):
    t0 = time.perf_counter(); out = fn(); times[name] = time.perf_counter() - t0; return out

pre = step("normalise + variable genes", lambda: scfast.preprocess(counts, n_hvg=2000))
hvg = pre["data"][:, pre["hvg"]].tocsr(); hvg.sort_indices()
emb, _, _ = step("PCA (30 components)", lambda: scfast.pca(hvg, n_comps=30))
idx, dist = step("20 nearest neighbours", lambda: scfast.knn(np.ascontiguousarray(emb[:, :10]), k=20))
graph = step("shared-neighbour graph", lambda: scfast.snn(idx))
labels = step("Leiden clustering", lambda: scfast.leiden(graph, resolution=0.5))
xy = step("UMAP", lambda: scfast.umap(idx, dist, init=np.ascontiguousarray(emb[:, :2])))

print(f"{counts.shape[0]} cells x {counts.shape[1]} genes, {labels.max() + 1} clusters")
for k, v in times.items():
    print(f"  {k:<28} {v:6.2f} s")
print(f"  {'total':<28} {sum(times.values()):6.2f} s")
plt.figure(figsize=(5, 5)); plt.scatter(xy[:, 0], xy[:, 1], c=labels, s=3, cmap="tab20")
plt.title("PBMC 3k, scfast Leiden clusters"); plt.axis("off"); plt.show()

## Part 2: run the whole pipeline (optional, about 25–35 minutes)

This installs R 4 with Seurat 5 and Bioconductor as Ubuntu binaries, using
[r2u](https://github.com/eddelbuettel/r2u), so nothing compiles. It then installs CellWarden
from its release.

**Review agents (optional):** add an API key in Colab's **Secrets** panel (the key icon on the
left), named `ANTHROPIC_API_KEY`, `OPENAI_API_KEY` or `NVIDIA_API_KEY`, and give this notebook
access. NVIDIA's [build.nvidia.com](https://build.nvidia.com) offers free development credits.
Without a key, the gates are skipped and the rules decide.

In [ ]:
%%bash
# R 4.x + CRAN/Bioconductor packages as Ubuntu binaries (r2u), about 10 minutes
set -e
curl -fsSL https://raw.githubusercontent.com/eddelbuettel/r2u/master/inst/scripts/add_cranapt_jammy.sh | bash > /tmp/r2u.log 2>&1
apt-get install -y -qq r-cran-seurat r-cran-seuratobject r-cran-hdf5r r-cran-jsonlite r-cran-digest r-cran-yaml \
  r-cran-optparse r-cran-data.table r-cran-matrix r-cran-irlba r-cran-harmony r-cran-uwot r-cran-leidenbase \
  r-cran-qs2 r-cran-rhpcblasctl r-cran-remotes r-bioc-deseq2 r-bioc-limma r-bioc-edger r-bioc-mast \
  r-bioc-glmgampoi r-bioc-scdblfinder r-bioc-summarizedexperiment > /tmp/r-packages.log 2>&1
Rscript -e 'cat("R", R.version$major, R.version$minor, "| Seurat", format(packageVersion("Seurat")), "\n")'

In [ ]:
CW_VERSION = "0.3.0"
RELEASE = f"https://github.com/cellwarden/cellwarden/releases/download/v{CW_VERSION}"
!test -d cellwarden || git clone -q --depth 1 --branch v{CW_VERSION} https://github.com/cellwarden/cellwarden.git
!pip -q install "{RELEASE}/scfast-{CW_VERSION}-cp310-abi3-manylinux_2_28_x86_64.whl" "./cellwarden/python[harmony,report,llm]"
import os
os.environ["CELLWARDEN_REPO_ROOT"] = os.path.abspath("cellwarden")   # R stages run from the cloned repository
os.environ["CELLWARDEN_THREADS"] = str(os.cpu_count())

In [ ]:
import os
have_key = False
try:
    from google.colab import userdata
    for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "NVIDIA_API_KEY"):
        try:
            v = userdata.get(k)
        except Exception:
            v = None
        if v:
            os.environ[k] = v; have_key = True
except ImportError:
    have_key = any(os.environ.get(k) for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "NVIDIA_API_KEY"))
print("review agents:", "on" if have_key else "off (no key in Colab Secrets): the rules decide, gates are skipped")
!cellwarden doctor | sed -n '/^R:/,/^engine:/p;/^llm:/,/^$/p'

In [ ]:
# The whole pipeline on PBMC 3k: about 10-15 minutes on Colab's 2 CPUs
cfg = "cellwarden/examples/pbmc3k/config.yaml" if have_key else "cellwarden/examples/pbmc3k/config_no_llm.yaml"
extra = "" if have_key else "--llm none --no-gates"
!cellwarden run filtered_gene_bc_matrices/hg19 --config {cfg} {extra} --name pbmc3k --out pbmc3k_colab --progress plain

In [ ]:
from IPython.display import HTML, display
report = "pbmc3k_colab/S5/report.html"
display(HTML(f"<p>Your report: <code>{report}</code> (download it from the Files panel on the left, "
             "or open it below)</p>"))
display(HTML(open(report).read()))

## Next steps

- **Your own data:** `cellwarden run data/*.h5 -i --out results/study`. With `-i` it asks a few
  questions first; Enter keeps the defaults.
- **Install on your machine:** `docker pull ghcr.io/cellwarden/cellwarden` or
  `apptainer pull docker://ghcr.io/cellwarden/cellwarden`, then see
  [Getting started](https://github.com/cellwarden/cellwarden/blob/main/docs/GETTING_STARTED.md).
- **Tell us how it went:** [I tested CellWarden](https://github.com/cellwarden/cellwarden/issues/new?template=tested.yml).